In [1]:
import pandas as pd
import numpy as np

In [2]:
import pandas as pd
from transformers import GPT2LMHeadModel, GPT2Tokenizer, TextDataset, DataCollatorForLanguageModeling, Trainer, TrainingArguments

# Load the CSV file containing disease-drug pairs
data = pd.read_excel("/kaggle/input/medicine-recommendation/Medicine_description.xlsx")
data

2024-02-26 22:54:06.865118: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-02-26 22:54:06.865221: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-02-26 22:54:07.004030: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


,Drug_Name,Reason,Description
0,A CN Gel(Topical) 20gmA CN Soap 75gm,Acne,Mild to moderate acne (spots)
1,A Ret 0.05% Gel 20gmA Ret 0.1% Gel 20gmA Ret 0...,Acne,A RET 0.025% is a prescription medicine that i...
2,ACGEL CL NANO Gel 15gm,Acne,It is used to treat acne vulgaris in people 12...
3,ACGEL NANO Gel 15gm,Acne,It is used to treat acne vulgaris in people 12...
4,Acleen 1% Lotion 25ml,Acne,treat the most severe form of acne (nodular ac...
...,...,...,...
22476,T Muce Ointment 5gm,Wound,used for treating warts
22477,Wokadine 10% Solution 100mlWokadine Solution 5...,Wound,used to soften the skin cells
22478,Wokadine M Onit 10gm,Wound,used for scars
22479,Wound Fix Solution 100ml,Wound,used for wounds


In [3]:

# Preprocess the data to create text examples in the format "Disease: Drug"
text_examples = data["Reason"] + ": " + data["Drug_Name"]

# Write the preprocessed text examples to a text file
with open("medical_text_data.txt", "w") as f:
    for example in text_examples:
        f.write(example + "\n")



In [4]:
# Load the pre-trained GPT-2 tokenizer and LM model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Load the preprocessed text data and tokenize it
train_dataset = TextDataset(tokenizer=tokenizer, file_path="/kaggle/working/medical_text_data.txt", block_size=128)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./LM_output",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=1000,
    save_total_limit=2,
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    train_dataset=train_dataset,
)

# Fine-tune the LM on medical data
trainer.train()

# Save the fine-tuned model
trainer.save_model("./fine_tuned_medical_model")


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

  ········································


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Step,Training Loss
500,1.700100
1000,1.448500
1500,1.362800
2000,1.310500


In [9]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load the fine-tuned GPT-2 LM model
model = GPT2LMHeadModel.from_pretrained("/kaggle/working/fine_tuned_medical_model/")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Input text
input_text = "Depression"

# Tokenize input text
input_ids = tokenizer.encode(input_text, return_tensors="pt")

# Generate output text using the LM
output = model.generate(input_ids, max_length=100, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)

# Decode and print the generated text
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print("Generated text:", generated_text)


Generated text: Depression: Zolpraz 0.5mg Tablet 10'SZolpraz 0.25mg Tablet 10'S
Depression: Zolpraz Plus 0.5mg Tablet 10'SZolpraz Plus 0.25mg Tablet 10'S
Depression: Zolpraz SR 0.5mg Tablet 10'SZolpraz SR 0.25mg Tablet 10'S
Depression: Zolpraz SR 0.25mg


In [11]:
from IPython.display import FileLink

# Specify the path to the file you want to download
file_path = "fine_tuned_medical_model/model.safetensors"

# Create a download link for the file
FileLink(file_path)


/kaggle/working/fine_tuned_medical_model/model.safetensors